# MAINTAIN AI V1.3 — Colab Dataset Bootstrap

This notebook keeps large raw datasets inside the temporary Colab runtime.

Automatically acquired: C-MAPSS/MetroPT-3/pump can remain in the existing V1.2 flow; this notebook adds CWRU + Paderborn for the induction_motor domain.

Important: CWRU and Paderborn are bearing-condition/fault datasets, not timestamped 24h/48h/7d run-to-failure datasets. They are therefore used for motor-domain fault/representation learning, not fabricated future-risk labels.


In [ ]:
!pip -q install requests beautifulsoup4 scipy pandas numpy scikit-learn torch pyarrow


In [ ]:
import os,re,json,subprocess
from pathlib import Path
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import numpy as np,pandas as pd
from scipy.io import loadmat

ROOT=Path("/content/maintain_ai_v1_3")
RAW=ROOT/"raw"; RAW.mkdir(parents=True,exist_ok=True)
print("Runtime:",ROOT)


# 1. Download Paderborn automatically

The official Paderborn directory contains the 32 bearing archives: 6 healthy, 12 artificially damaged and 14 real-damage states. The archives are large, so do not download them to Windows.


In [ ]:
PADERBORN_BASE="https://groups.uni-paderborn.de/kat/BearingDataCenter/"
PADERBORN_CODES=[
"K001","K002","K003","K004","K005","K006",
"KA01","KA03","KA04","KA05","KA06","KA07","KA08","KA09","KA15","KA16","KA22","KA30",
"KB23","KB24","KB27",
"KI01","KI03","KI04","KI05","KI07","KI08","KI14","KI16","KI17","KI18","KI21"]

def download(url,dst):
    dst=Path(dst); dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists() and dst.stat().st_size>0:
        print("SKIP",dst.name,round(dst.stat().st_size/1e6,1),"MB"); return
    print("GET",url)
    with requests.get(url,stream=True,timeout=180) as r:
        r.raise_for_status()
        with open(dst,"wb") as f:
            for chunk in r.iter_content(8*1024*1024):
                if chunk: f.write(chunk)
    print("DONE",dst.name,round(dst.stat().st_size/1e6,1),"MB")

PRAW=RAW/"paderborn_archives"; PRAW.mkdir(exist_ok=True)
for c in PADERBORN_CODES:
    download(urljoin(PADERBORN_BASE,c+".rar"),PRAW/(c+".rar"))
print("Paderborn archives:",len(list(PRAW.glob("*.rar"))))


# 2. Extract Paderborn inside Colab

Nothing is written to your Windows Downloads folder. The temporary Colab disk is used instead.


In [ ]:
subprocess.run(["bash","-lc","apt-get -qq update && apt-get -qq install -y p7zip-full"],check=True)
PMAT=RAW/"paderborn_mat"; PMAT.mkdir(exist_ok=True)
for rar in sorted(PRAW.glob("*.rar")):
    marker=PMAT/(rar.stem+".done")
    if marker.exists(): continue
    subprocess.run(["7z","x","-y",str(rar),f"-o{PMAT}"],check=True)
    marker.write_text("ok")
print("Paderborn MAT files:",len(list(PMAT.rglob("*.mat"))))


# 3. Download CWRU MATLAB files automatically

The official CWRU pages expose MATLAB files as links. This cell parses those pages and downloads the normal baseline plus the complete 12 kHz drive-end fault set directly to Colab.

You never need to open a MATLAB file in Chrome.


In [ ]:
CWRU_NORMAL="https://engineering.case.edu/bearingdatacenter/normal-baseline-data"
CWRU_12K="https://engineering.case.edu/bearingdatacenter/12k-drive-end-bearing-fault-data"

def mat_links(page):
    html=requests.get(page,timeout=90).text
    soup=BeautifulSoup(html,"html.parser")
    out={}
    for a in soup.find_all("a",href=True):
        href=urljoin(page,a["href"])
        if href.lower().split("?")[0].endswith(".mat"):
            label=a.get_text(" ",strip=True) or Path(href).stem
            label=re.sub(r"[^A-Za-z0-9@._-]+","_",label).strip("_")
            if not label.lower().endswith(".mat"): label+=".mat"
            out[label]=href
    return out

links={**mat_links(CWRU_NORMAL),**mat_links(CWRU_12K)}
print("CWRU files discovered:",len(links))
CWRU=RAW/"cwru"; CWRU.mkdir(exist_ok=True)
for name,url in sorted(links.items()):
    download(url,CWRU/name)
json.dump({"files":sorted(links)},open(ROOT/"cwru_manifest.json","w"),indent=2)
print("CWRU downloaded:",len(list(CWRU.glob("*.mat"))))


# 4. Verify both datasets

This does not train yet. It verifies that Colab actually received readable MATLAB files before GPU preprocessing/training.


In [ ]:
p=sorted(PMAT.rglob("*.mat"))
c=sorted(CWRU.glob("*.mat"))
assert p,"No Paderborn .mat files found"
assert c,"No CWRU .mat files found"

pm=loadmat(str(p[0]),squeeze_me=True,struct_as_record=False)
cm=loadmat(str(c[0]),squeeze_me=True,struct_as_record=False)
print("Paderborn:",p[0].name)
print([k for k in pm if not k.startswith("__")])
print("CWRU:",c[0].name)
print([k for k in cm if not k.startswith("__")])


# 5. Save provenance

The raw multi-GB data stays in Colab. Later training cells should export only model weights, normalization statistics, metrics and schema.


In [ ]:
manifest={
"version":"shared-industrial-v1.3",
"motor_category":"induction_motor",
"paderborn_archives":PADERBORN_CODES,
"cwru_files":len(c),
"tasks":{
"compressor_pump":["future_risk_24h","future_risk_48h","future_risk_7d"],
"induction_motor":["bearing_condition_classification","representation_pretraining"]},
"raw_runtime_dir":str(RAW)}
json.dump(manifest,open(ROOT/"manifest.json","w"),indent=2)
print(json.dumps(manifest,indent=2))
print("\nNext: run the motor feature/sequence training stage after acquisition succeeds.")
